Persoonlijk logboek Noah

Eerst zorg ik ervoor dat Mohammed zijn csv blijft bestaan en dat ik hem niet direcht overschrijft zodat mocht ik een fout maken in mijn csv er altijd nog bij een versie ervoor dat ongedaan kan maken.

In [1]:
import pandas as pd

df_original = pd.read_csv("../../data/processed/df_v5_clean_final.csv", sep=",")
df = df_original.copy()

Stap is van fase 1 is missende waarden analyseren en behandelen: Waarom doen we dit? Omdat missende waarden kunnen analyses vertekenen en zorgen voor onbetrouwbare conclusies.
Voor tijdreeksanalyse is het essentieel dat de dataset zo volledig mogelijk is.
Daarom controleren we eerst hoeveel missende waarden aanwezig zijn.

In [2]:
df.isna().sum()

Peildatum           0
Wijk                0
AantalInwoners_5    0
k_0Tot15Jaar_8      0
k_15Tot25Jaar_9     0
dtype: int64

df.Insa geeft alle missende waardes zoals je kan zien staat er overal 0 dat betekend dat we niks missen en door kunnen gaan naar mijn 2e stap van fase 1 en dat is Controleren of combinaties Wijk + Jaar uniek zijn. Waarom is dit belangrijk?

Voor tijdreeksanalyse moet iedere wijk per jaar maar één record hebben.
Als een wijk in hetzelfde jaar meerdere keren voorkomt:

kan dit leiden tot foutieve analyses.

worden gemiddelden verkeerd berekend.

ontstaan er dubbele tellingen.

Daarom controleren we of de combinatie Wijk + Jaar uniek is.


In [3]:
# Controleren op dubbele combinaties van Wijk en Peildatum
duplicates_wijk_datum = df.duplicated(subset=["Wijk", "Peildatum"])

# Aantal dubbele combinaties tellen
aantal_dubbele_combinaties = duplicates_wijk_datum.sum()

print("Aantal dubbele combinaties van Wijk + Peildatum:")
print(aantal_dubbele_combinaties)

Aantal dubbele combinaties van Wijk + Peildatum:
0


Uit de controle die we hebben uitgevoerd blijkt dus dat er geen dubbele combinasities zijn van wijken en de peildatum. We gaan verder naar stap 3 kijken of er dubbele records zijn. In deze stap controleren we of er volledig identieke rijen in de dataset voorkomen.

Dit betekent dat alle kolommen exact dezelfde waarden bevatten.

Waarom is dit belangrijk?

Volledig dubbele records kunnen ontstaan door:

Fouten bij het samenvoegen van datasets

Onjuiste export uit een databron

Meerdere keren opslaan van dezelfde observatie

Als deze niet worden verwijderd, kunnen ze:

Resultaten vertekenen

Totalen en gemiddelden beïnvloeden

De betrouwbaarheid van de analyse verminderen

Daarom controleren we of de dataset volledig unieke rijen bevat.

In [4]:
# Controleren op volledig dubbele rijen
aantal_volledig_dubbel = df.duplicated().sum()

print("Aantal volledig dubbele records:")
print(aantal_volledig_dubbel)

Aantal volledig dubbele records:
0


We hebben een test gerund en zoals te zien is zijn er geen dubbele records dus gaan we weer door naar stap 4:In deze stap controleren we of er veranderingen zijn in wijknamen of wijkstructuur over de tijd.

Bij CBS-data kunnen soms:

Wijken hernoemd worden

Wijken samengevoegd worden

Nieuwe wijken ontstaan

Als dit gebeurt, kan dat analyses verstoren.
Bijvoorbeeld: een wijk lijkt ineens te groeien terwijl het eigenlijk een samenvoeging is.

Daarom controleren we of:

Wijknamen consistent zijn

Geen rare wijzigingen over tijd voorkomen

In [5]:
print("Aantal unieke wijken:")
print(df["Wijk"].nunique())

Aantal unieke wijken:
56


In [6]:
df.groupby("Peildatum")["Wijk"].nunique()

Peildatum
2018-01-01     5
2019-01-01     5
2020-01-01     5
2021-01-01     5
2022-01-01    53
2023-01-01    54
2024-01-01    55
2025-01-01    55
Name: Wijk, dtype: int64

Uit de test blijkt dat er 56 Unieke wijken zijn maar als je kijkt per peildatum dar er in 2018 tot 21 maar 5 zijn en dan 53-54-55-55.

Fase 3 Noah: Voor modellering  mogen meestal alleen numerieke variabelen gebruikt worden.

Niet-numerieke kolommen zoals:

Wijknamen (tekst)

Datums

Eventuele categorieën

kunnen een model verstoren of errors geven.

Daarom:

1️ We behouden alleen numerieke kolommen
2️ We verwijderen overbodige variabelen
3️ We controleren nog een keer op missende waarden
4️ We maken een definitieve dataset

In [7]:

import pandas as pd
df = pd.read_csv("../../data/processed/df_v10_ratio_temp.csv")
df.head()

df.dtypes

Peildatum               str
Wijk                    str
AantalInwoners_5      int64
k_0Tot15Jaar_8        int64
k_15Tot25Jaar_9       int64
Jaar                  int64
pct_0_15            float64
pct_15_25           float64
Ratio_0_15          float64
Ratio_15_25         float64
dtype: object

Zoals te zien is zijn peildatum en wijk niet numeriek en dat gaan we omzetten naar numerieke waarden.

In [8]:
# Peildatum omzetten naar datetime zodat we peildatum numeriek kunnen maken
df['Peildatum'] = pd.to_datetime(df['Peildatum'])

# Jaar extraheren
df['jaar'] = df['Peildatum'].dt.year

Nu hebben we van peildatum "jaar" gemaakt met een numerieke waarden nu kunnen we de oude verie deleten oftewel peildatm en dit gaan we gelijk controleren

In [9]:
df = df.drop(columns=['Peildatum'])

In [22]:
#controleren of peildatum weg is en de numerieke vervaning jaar ook daadwerkelijk numeriek is
df.dtypes

Wijk                    str
Jaar                  int64
AantalInwoners_5      int64
k_0Tot15Jaar_8        int64
k_15Tot25Jaar_9       int64
Ratio_0_15          float64
Ratio_15_25         float64
Bevolkingsgroei     float64
dtype: object

Nu dit klopt gaan we door en kijken of er nog 0 waardes zijn die NaN waardes willen we niet hebben doordat het analyses onbetrouwbaar kan maken. 

In [11]:
# isna() markeert elke cel als True (bij NaN) of False, en sum() telt vervolgens alle True waardes op per kolom.
df.isna().sum()

Wijk                0
AantalInwoners_5    0
k_0Tot15Jaar_8      0
k_15Tot25Jaar_9     0
Jaar                0
pct_0_15            0
pct_15_25           0
Ratio_0_15          0
Ratio_15_25         0
jaar                0
dtype: int64

HIer mee concluderen we dat er geen NaN waardes zijn

In [12]:
df.to_csv("df_v11_peildatum_numeriek.csv", index=False)

Fase 5 gaan we dit uitvoeren :•	- Is R² voldoende?
•	- Wat zijn beperkingen van de dataset?
•	- Is de tijdreeks lang genoeg?
•	- Heeft wijkwijziging invloed?
 
 waarbij we de relatie leggen bij dus de bevolkingsgroei en het aantal niuewbouw woningen per wijk.

In [ ]:
# bestand inlezen
df = pd.read_csv("../../data/processed/df_plot_1.csv", sep=";")

df.head()

,Stadsdeel,2018,2021,groei,percentage_groei,totaal_nieuwbouw
0,Buiten,11190,10480,-710,-6.344951,1127.0
1,Haven,3975,4180,205,5.157233,396.0
2,Hout,465,1570,1105,237.634409,1715.0
3,Poort,3375,4585,1210,35.851852,2427.0
4,Stad,19915,18890,-1025,-5.146874,416.0


In [ ]:
#Kijken welke colommen we hebben
df.columns

Index(['Stadsdeel', '2018', '2021', 'groei', 'percentage_groei',
       'totaal_nieuwbouw'],
      dtype='object')

In [ ]:
# we gaan een verband leggen tussen de nieuwbouw en bevolkoingsgroei 0 tot 15
correlatie = df["totaal_nieuwbouw"].corr(df["groei"])

print("r =", correlatie)

r = 0.7687911313830859


In [51]:
r_kwadraat = correlatie ** 2

print("R² =", r_kwadraat)

R² = 0.5910398036932851


In [55]:
df = pd.read_csv("../../data/processed/df_plot_2.csv", sep=";")

df.head()

,Stadsdeel,2018,2021,groei,percentage_groei,totaal_nieuwbouw
0,Buiten,7730,7780,50,0.646831,1127.0
1,Haven,2420,2465,45,1.859504,396.0
2,Hout,235,555,320,136.170213,1715.0
3,Poort,1220,1600,380,31.147541,2427.0
4,Stad,14890,14625,-265,-1.779718,416.0


In [56]:
correlatie = df["totaal_nieuwbouw"].corr(df["groei"])

print("r =", correlatie)

r = 0.8806099281683414


In [57]:
r_kwadraat = correlatie ** 2

print("R² =", r_kwadraat)

R² = 0.7754738455886514


Nu we hebben gekeken wat R² is kunnen we concluderen dat er een redelijk sterk verband is tussen bevolkingsgroei en Niuewbouw voor de leeftijds catogorie 0-15 en we zien dar er een zeer sterk verband is voor de leeftijds catogorie 15-25 maar hoewel R² een redelijk sterk verband laat zien tussen nieuwbouw en bevolkingsgroei is dit niet voldoende om een oorzakelijk verband te bewijzen Andere factoren zoals migratie of economische ontwikkelingen kunnen ook invloed hebben op de bevolkingsgroei.

Dat leid ons naar vraag 2 van Fase 5 Wat zijn beperkingen van de dataset? Het model bevat alleen bevolkingsgroei en nieuwbouw maar bevolkingsgroei kan komen door meerdere factoren zijn de huizenpijsen beter? is veel sprake van migratie in dat specifieke stadsdeel? dus omdanks dat er wel sprake is van een sterk verband kunnen er toch nog externe factoren bij kijken die verder onderzoek nodig hebben

is de tijdreeks lang genoeg?

Vraag 4 hebben eventuele wijkwijzingen invloed op de analyse? jazeker wijkwijzigingen kunnen invloed hebben op de analyse. Wanneer wijkgrenzen veranderen of wijken worden samengevoegd kan dit leiden tot veranderingen in bevolkingsaantallen die niet direct verband houden met nieuwbouw.